# W06 課堂練習：向量、BM25、RRF 混合檢索比較（20 分鐘）

切塊與嵌入沿用 W05 的「結構＋標題」，只換檢索方法，用 16 題評測集比較四種設定的 Recall@5 與 MRR。

**做法**：由上往下依序執行每個儲存格（`Shift + Enter`）。第一個儲存格需下載模型，約 1 分鐘。

In [ ]:
# 1. 安裝套件、讀取講義、切塊（與 W05 結構＋標題相同）、建向量索引
!pip install -q sentence-transformers jieba

import re, math, urllib.request
from collections import Counter
import numpy as np, pandas as pd
import jieba
from sentence_transformers import SentenceTransformer

URL = "https://raw.githubusercontent.com/NCU-AIKslab/ncu-1151-creative-learning-final/main/data/textbook/linear-equation.md"
md = urllib.request.urlopen(URL).read().decode("utf-8")
model = SentenceTransformer("BAAI/bge-small-zh-v1.5")

sections, chapter, section, cur = [], None, None, None
for line in md.splitlines():
    if line.startswith("# "):
        chapter = line[2:].strip()
    elif m := re.match(r"^## (\d+-\d+) (.+)", line):
        section, cur = f"{m[1]} {m[2]}", None
    elif line.startswith("## "):
        cur = None
    elif m := re.match(r"^### (\d+-\d+-\d+) (.+)", line):
        cur = {"id": m[1], "path": f"{chapter} > {section} > {m[2]}", "text": ""}
        sections.append(cur)
    elif cur is not None:
        cur["text"] += line + "\n"

chunks = [{"id": s["id"], "text": f"{s['path']}\n{s['text'].strip()}"} for s in sections]
vecs = model.encode([c["text"] for c in chunks], normalize_embeddings=True)
print(f"共 {len(chunks)} 塊")

## 2. BM25

對查詢中的每個詞 t：`IDF(t) × f × (k1 + 1) ÷ [f + k1 × (1 − b + b × 文件長度 ÷ 平均長度)]`，再加總。

中文要先**斷詞**，這裡用 jieba，另外：

- 把 `1-3`、`1-3-4` 這類代號當成一個詞（否則會被拆成 `1`、`-`、`3`）
- 把教材的專有名詞加進自訂辭典（`USE_USERDICT`）
- 去掉「的、是、怎麼」這類停用詞（`USE_STOPWORDS`）

In [ ]:
# 2. 斷詞 + BM25
K1, B = 1.2, 0.75
USE_USERDICT, USE_STOPWORDS = True, True
USERDICT = ["一元一次方程式", "代數式", "等量公理", "移項", "去括號", "去分母", "驗算",
            "雞兔同籠", "未知數", "同類項", "最小公倍數", "分配律", "應用問題"]
STOPWORDS = set("的 是 在 有 要 和 又 它 嗎 呢 了 會 再 先 把 用 時 就 這 那 一 個 "
                "怎麼 什麼 哪些 為什麼 能 不能 意思 如果 例如 可以 稱為".split())

jieba.setLogLevel(60)
for w in USERDICT:
    jieba.add_word(w) if USE_USERDICT else jieba.del_word(w)

def tokenize(text):
    text = text.lower()
    codes = re.findall(r"\d+(?:-\d+)+", text)          # 章節代號整個保留
    rest = re.sub(r"\d+(?:-\d+)+", " ", text)
    words = [w for w in jieba.lcut(rest) if re.search(r"\w", w)]  # 去掉空白與標點
    if USE_STOPWORDS:
        words = [w for w in words if w not in STOPWORDS]
    return codes + words

docs = [tokenize(c["text"]) for c in chunks]
N, avgdl = len(docs), sum(map(len, docs)) / len(docs)
df = Counter(t for d in docs for t in set(d))
idf = {t: math.log((N - n + 0.5) / (n + 0.5) + 1) for t, n in df.items()}

def bm25_scores(q):
    scores = np.zeros(N)
    for i, d in enumerate(docs):
        tf = Counter(d)
        for t in tokenize(q):
            f = tf[t]
            if f:
                scores[i] += idf[t] * f * (K1 + 1) / (f + K1 * (1 - B + B * len(d) / avgdl))
    return scores

print("斷詞示範：", tokenize("1-3 節的移項和等量公理有什麼關係？"))

## 3. 四種設定

| 設定 | 做法 |
| --- | --- |
| 純向量 | 餘弦相似度排序 |
| 純 BM25 | BM25 分數排序；分數為 0 的不上榜 |
| RRF 混合 | 向量與 BM25 各取前 10 名，`RRF(d) = Σ 1 / (k + rank)`，k = 60 |
| 加權相加（錯誤示範） | `0.5 × 餘弦相似度 + 0.5 × BM25 分數` |

先用投影片的算例（A–G）驗證 `rrf()` 寫對了。

In [ ]:
# 3. 檢索函式與 RRF
RRF_K, CAND = 60, 10

def vector_rank(q):
    qv = model.encode([q], normalize_embeddings=True)[0]
    return [chunks[i]["id"] for i in np.argsort(-(vecs @ qv))]

def bm25_rank(q):
    s = bm25_scores(q)
    return [chunks[i]["id"] for i in np.argsort(-s) if s[i] > 0]   # 沒命中任何詞的不上榜

def naive_rank(q):
    qv = model.encode([q], normalize_embeddings=True)[0]
    s = 0.5 * (vecs @ qv) + 0.5 * bm25_scores(q)
    return [chunks[i]["id"] for i in np.argsort(-s)]

def rrf(*rankings, k=RRF_K):
    score = Counter()
    for ranking in rankings:
        for r, doc in enumerate(ranking, 1):
            score[doc] += 1 / (k + r)
    return [doc for doc, _ in score.most_common()]

def hybrid_rank(q):
    return rrf(vector_rank(q)[:CAND], bm25_rank(q)[:CAND])

SETTINGS = {"純向量": vector_rank, "純 BM25": bm25_rank, "RRF 混合": hybrid_rank, "加權相加": naive_rank}

# 投影片算例：向量 A B C D E、BM25 C F B A G → 融合後應為 C A B F D …
print(rrf(list("ABCDE"), list("CFBAG")))

## 4. 評測：Recall@5 與 MRR

- **Recall@5**：正確塊有幾成出現在前 5 名
- **MRR**：第一個正確塊名次的倒數，取平均

In [ ]:
# 4. 評測集與指標
EVAL = [
    # 精確：代號、專有名詞、原文片語
    ("精確",   "1-3 節有哪些例題？",                          {"1-3-3", "1-3-4"}),
    ("精確",   "雞兔同籠怎麼列式？",                          {"1-5-4"}),
    ("精確",   "等量除法公理有什麼限制？",                      {"1-2-2"}),
    ("精確",   "3(x + 2) − 2(x − 1) = 15 怎麼展開？",          {"1-4-1"}),
    ("精確",   "天平的比喻是在說明什麼？",                      {"1-2-1"}),
    ("精確",   "x² + 1 = 5 為什麼不是一元一次方程式？",          {"1-1-3"}),
    # 語意：刻意換掉原文用字
    ("語意",   "把數字搬到等號另一邊，正負號要怎麼處理？",          {"1-3-1"}),
    ("語意",   "怎麼確定算出來的答案沒有錯？",                    {"1-3-5"}),
    ("語意",   "題目裡有分數，能不能先變成整數再算？",              {"1-4-2"}),
    ("語意",   "學生最容易在哪些地方算錯？",                      {"1-4-4"}),
    ("語意",   "爸爸和小孩幾年後歲數會變成兩倍，要怎麼算？",          {"1-5-2"}),
    ("語意",   "用不同速度上學，怎麼反推家到學校多遠？",            {"1-5-5"}),
    # 跨段落：答案要兩塊才完整
    ("跨段落", "移項和等量公理有什麼關係？",                      {"1-3-1", "1-2-1"}),
    ("跨段落", "乘除的移項是根據哪一條公理？",                    {"1-3-2", "1-2-2"}),
    ("跨段落", "方程式裡同時有括號和分數，要怎麼處理？",            {"1-4-1", "1-4-2"}),
    ("跨段落", "什麼是方程式？它的解又是什麼意思？",                {"1-1-2", "1-1-4"}),
]

def recall_at(ranking, gold, k=5):
    return len(set(ranking[:k]) & gold) / len(gold)

def rr(ranking, gold):
    return next((1 / r for r, doc in enumerate(ranking, 1) if doc in gold), 0)

rows = []
for typ, q, gold in EVAL:
    for name, fn in SETTINGS.items():
        ranking = fn(q)
        rows.append({"類型": typ, "問題": q, "設定": name,
                     "Recall@5": recall_at(ranking, gold), "RR": rr(ranking, gold)})
df_eval = pd.DataFrame(rows)

summary = df_eval.pivot_table(index="設定", columns="類型", values=["Recall@5", "RR"], aggfunc="mean")
summary[("Recall@5", "整體")] = df_eval.groupby("設定")["Recall@5"].mean()
summary[("RR", "整體")] = df_eval.groupby("設定")["RR"].mean()
summary = summary.rename(columns={"RR": "MRR"}, level=0).reindex(list(SETTINGS)).round(2)
summary[[(m, t) for m in ["Recall@5", "MRR"] for t in ["整體", "精確", "語意", "跨段落"]]]

## 5. 逐題看：誰救了誰

每題第一個正確塊的名次（❌ 表示整份排名都沒出現）。找出「向量失手、BM25 救回來」與反過來的題目。

In [ ]:
# 5. 逐題名次
def first_hit(ranking, gold):
    return next((r for r, doc in enumerate(ranking, 1) if doc in gold), None)

table = pd.DataFrame([
    {"類型": typ, "問題": q, **{name: (f"第 {r} 名" if (r := first_hit(fn(q), gold)) else "❌")
                               for name, fn in SETTINGS.items()}}
    for typ, q, gold in EVAL])
table

## 6. 錯誤示範：為什麼加權相加會變成「比較慢的 BM25」

同一題，比較兩種檢索方法的**原始分數**尺度：相加後誰說了算？

In [ ]:
# 6. 分數尺度比較
Q = "移項和等量公理有什麼關係？"

qv = model.encode([Q], normalize_embeddings=True)[0]
cos, bm = vecs @ qv, bm25_scores(Q)
print(f"餘弦相似度：最高 {cos.max():.2f}、最低 {cos.min():.2f}")
print(f"BM25 分數 ：最高 {bm.max():.2f}、最低 {bm.min():.2f}")

naive = 0.5 * cos + 0.5 * bm
show = lambda s: [chunks[i]["id"] for i in np.argsort(-s)[:5]]
pd.DataFrame({"純向量": show(cos), "純 BM25": show(bm), "0.5 × 向量 + 0.5 × BM25": show(naive),
              "RRF 混合": hybrid_rank(Q)[:5]}, index=range(1, 6))

## 7. 動一個參數

回到第 2 或第 3 步，**只改一個**，再從該步往下重跑，看第 4 步的表格怎麼變：

- `B = 0`（不管文件長度）
- `USE_USERDICT = False`（不加自訂詞）
- `USE_STOPWORDS = False`（不去停用詞）
- `RRF_K = 0` 或 `RRF_K = 1000`

## 8. 回報

個人填寫[表單](https://forms.gle/1zdU2NqbCieBVzyL6)。第 1、2 題用預設參數的結果回答，第 3 題才改參數。

1. 第 4 步的表格：RRF 混合與加權相加，各自在哪一類題目的哪個指標上，比向量和 BM25 兩種檢索方法中較好的那一個還低？
2. 從第 5 步挑一題向量與 BM25 名次差很多的題目，用第 2 步的斷詞結果或講義內容說明原因
3. 第 7 步改了哪一個參數？改之前你預測會怎樣？實際上哪一類題目的哪個指標變了？為什麼？